[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Karang1908/bio-bot/blob/main/notebooks/bio_bot_colab_training.ipynb)

# bio-bot: Vision Approach Training on Google Colab (Free T4 GPU) - Round 16

**Objective**: Continue training the Go2 quadruped to discover visual approach ("come" and "ball") from rest in the park without teacher help (`help_ = 0.0`), using coordinated forward-turning arc velocity commands (`task_cmd`) with a forward velocity floor ($v_{\text{fwd}} \ge 0.20\text{ m/s}$), exact `arctan2` yaw error steering, and stillness zeroing upon target arrival.

- **Restart Checkpoint**: `baby_dog_step_0000491520.npz` (SHA256 `5a534b328c2c511d0cdccc105f61f9c12fdc5b767662f697af05315f06e16d17`, 3.12% arrival record)
- **Compute Budget**: $0 (Google Colab Free Tier, Nvidia Tesla T4 GPU)
- **Training Steps**: 1.5M steps (~91 updates), 3 evaluations (0.5M, 1.0M, 1.5M), 3 cohorts (192 lives per eval)

### Step 1: Verify Remote GPU

In [ ]:
!nvidia-smi

### Step 2: Install Pinned Dependencies
Matches the verified Kaggle/bio-bot software stack exactly.

In [ ]:
!pip install -q "jax[cuda12]==0.7.2" "jaxlib==0.7.2" "mujoco==3.14.0" "mujoco-mjx==3.14.0" optax

### Step 3: Clone bio-bot Repository

In [ ]:
import os
if not os.path.exists("bio-bot"):
    !git clone https://github.com/Karang1908/bio-bot.git
%cd bio-bot
!git pull origin main

### Step 4: Mount or Upload Restart Candidate Checkpoint
Upload `baby_dog_step_0000491520.npz` (3.9 MB) from your local `results/baby_dog_vision_approach_round10/` directory, or use Google Drive / Kaggle API.

In [ ]:
import os
import hashlib
from pathlib import Path

ckpt_dir = Path("checkpoints")
ckpt_dir.mkdir(exist_ok=True)
candidate_path = ckpt_dir / "baby_dog_step_0000491520.npz"

if not candidate_path.exists():
    print("Uploading candidate checkpoint...")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname, data in uploaded.items():
            if fname.endswith(".npz"):
                (ckpt_dir / fname).write_bytes(data)
                candidate_path = ckpt_dir / fname
    except Exception as e:
        print("Upload helper exception:", e)

EXPECTED_SHA256 = "5a534b328c2c511d0cdccc105f61f9c12fdc5b767662f697af05315f06e16d17"
if candidate_path.exists():
    actual = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
    print(f"Candidate path: {candidate_path}")
    print(f"Candidate SHA256: {actual}")
    assert actual == EXPECTED_SHA256, f"Hash mismatch: expected {EXPECTED_SHA256}, got {actual}"
    print("✅ Candidate verified successfully!")
else:
    print("⚠️ Checkpoint not found yet. Please ensure baby_dog_step_0000491520.npz is in checkpoints/")

### Step 5: Run Self-Learning Continuation (Round 16)
- 1.5M steps (~91 updates)
- Teacher movement help strictly 0.0 from rest
- All 11 nominal motor skills rehearsed
- Coordinated forward arc task_cmd with velocity floor and arrival stopping

In [ ]:
!python train/baby_dog.py \
    --stage practice \
    --core v2 \
    --readout vector \
    --lesson approach \
    --park \
    --throws \
    --tasks \
    --drives \
    --voice \
    --self_learning \
    --let_go \
    --absolute_drives \
    --protect_skills \
    --lr 5e-6 \
    --voice_lr 0.0003 \
    --vision_lr 0.0003 \
    --task_scale 800.0 \
    --task_pay 16.0 \
    --fwd_pay 16.0 \
    --face_pay 4.0 \
    --yaw_pay 0.0 \
    --reach_pay 25.0 \
    --steps 1500000 \
    --envs 512 \
    --evals 3 \
    --eval_repeats 3 \
    --seed 1 \
    --brain checkpoints/baby_dog_step_0000491520.npz \
    --expect_checkpoint "after 0.5M steps" \
    --out results/colab_round16

### Step 6: Download Results & Snapshots

In [ ]:
!ls -lh results/colab_round16/
!zip -r bio_bot_colab_round16_results.zip results/colab_round16/
from google.colab import files
files.download("bio_bot_colab_round16_results.zip")